# 03 — Table 2: Cross-architecture mechanism comparison at matched seed count

Reproduces Table 2 of the manuscript: the matched-seed GEARS audit that separates instability from driver identity.

**Manuscript reference:** Table 2; Results §3.5.

The table is **derived from raw data** in this notebook (it is not read from a pre-baked CSV). For each condition the mechanism label and top-LOO driver mode are computed for:
- GEARS at n=7 (seeds 1-7),
- CPA matched at n=7 (CPA filtered to seeds 1-7),
- CPA n=100 (context column).

This makes the matched-n verdict reproducible from first principles without relying on the canonical `gears_cpa_mechanism_comparison.md` paper trail.

**Mechanism label rule** (Methods §2.6):
- `single-driver` if median LOO_max > 0.15 AND the same perturbation appears as top driver in more than 30% of high-LOO_max holdouts;
- `distributed` if median LOO_max < 0.10 AND no perturbation recurs above the same threshold;
- `mixed` otherwise.

**Sources:**
- `precomputed/eval/diag_loo_sensitivity_n100.csv` (CPA n=100 and matched seeds 1-7 subset).
- `precomputed/eval/diag_loo_sensitivity_gears.csv` (GEARS n=7).

**Outputs:**
- `precomputed/tables/table2_gears_matched_n.csv`


In [1]:
import sys
from pathlib import Path
from collections import Counter
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from perturb_style import PRECOMPUTED_DIR


## Load data

In [2]:
cpa_all = pd.read_csv(PRECOMPUTED_DIR / "eval" / "diag_loo_sensitivity_n100.csv")
cpa_all = cpa_all[cpa_all.model_id == "CPA"].copy()
gears = pd.read_csv(PRECOMPUTED_DIR / "eval" / "diag_loo_sensitivity_gears.csv")

print(f"CPA n=100 rows: {len(cpa_all)}")
print(f"GEARS rows: {len(gears)}")
print("\nGEARS condition counts:")
print(gears.groupby(["cell_type", "split_type"]).size().to_string())


CPA n=100 rows: 400
GEARS rows: 28

GEARS condition counts:
cell_type  split_type
K562       random        7
           stratified    7
RPE1       random        7
           stratified    7


## Mechanism label function

Applies the condition-level mechanism rule from Methods §2.6 to a slice of per-seed LOO data.

In [3]:
HIGH_LOO_THRESHOLD = 0.15  # operational threshold for single-driver flag
LOW_LOO_THRESHOLD  = 0.10  # threshold below which "distributed" is considered
RECURRENCE_FRAC    = 0.30  # fraction of high-LOO seeds in which mode must recur for "single-driver"

def mechanism_label_and_mode(slice_df):
    """Return (label, mode_driver, mode_count, n_high_loo, loo_max_median)."""
    loo_med = float(slice_df.LOO_max_abs_delta.median())
    high = slice_df[slice_df.LOO_max_abs_delta > HIGH_LOO_THRESHOLD]
    n_high = len(high)
    if n_high == 0:
        mode_driver, mode_count = None, 0
    else:
        c = Counter(high.top_loo_driver.dropna())
        if c:
            mode_driver, mode_count = c.most_common(1)[0]
        else:
            mode_driver, mode_count = None, 0
    mode_frac = (mode_count / n_high) if n_high > 0 else 0.0

    if loo_med > HIGH_LOO_THRESHOLD and mode_frac > RECURRENCE_FRAC:
        label = "single-driver"
    elif loo_med < LOW_LOO_THRESHOLD and mode_frac <= RECURRENCE_FRAC:
        label = "distributed"
    else:
        label = "mixed"
    return label, mode_driver, mode_count, n_high, loo_med


## Build the matched-n table

Compute three label/driver/LOO_max sets per condition: GEARS (n=7), CPA matched (seeds 1-7), CPA n=100 context.

In [4]:
CONDITIONS = [("K562", "random"), ("K562", "stratified"),
              ("RPE1", "random"), ("RPE1", "stratified")]

def render_cell(label, driver, loo_med):
    drv_str = driver if driver is not None else "-"
    return f"{label} ({drv_str}; {loo_med:.3f})"

rows = []
for cell, split in CONDITIONS:
    gears_slice = gears[(gears.cell_type == cell) & (gears.split_type == split)]
    g_lbl, g_drv, g_cnt, g_nh, g_loo = mechanism_label_and_mode(gears_slice)

    cpa_matched = cpa_all[(cpa_all.cell_type == cell)
                          & (cpa_all.split_type == split)
                          & (cpa_all.seed.between(1, 7))]
    m_lbl, m_drv, m_cnt, m_nh, m_loo = mechanism_label_and_mode(cpa_matched)

    cpa_n100 = cpa_all[(cpa_all.cell_type == cell) & (cpa_all.split_type == split)]
    n_lbl, n_drv, n_cnt, n_nh, n_loo = mechanism_label_and_mode(cpa_n100)

    verdict = "match" if g_lbl == m_lbl else "differ"
    rows.append({
        "Condition":           f"{cell} {split}",
        "GEARS (n=7)":         render_cell(g_lbl, g_drv, g_loo),
        "CPA matched (n=7)":   render_cell(m_lbl, m_drv, m_loo),
        "Verdict":             verdict,
        "CPA n=100":           render_cell(n_lbl, n_drv, n_loo),
    })

table2 = pd.DataFrame(rows)
print(table2.to_string(index=False))

agreement = (table2["Verdict"] == "match").sum()
print(f"\nMatched-n agreement: {agreement} of 4 conditions match.")


      Condition          GEARS (n=7)            CPA matched (n=7) Verdict                    CPA n=100
    K562 random  mixed (RPSA; 0.073)        mixed (POLR3A; 0.138)   match single-driver (MED12; 0.163)
K562 stratified mixed (SMC1A; 0.129) single-driver (MED17; 0.217)  differ single-driver (MED17; 0.163)
    RPE1 random  mixed (BET1; 0.092)             mixed (-; 0.102)   match         mixed (SF3B2; 0.088)
RPE1 stratified mixed (SF3B2; 0.076)         mixed (SF3B2; 0.067)   match         mixed (SF3B2; 0.092)

Matched-n agreement: 3 of 4 conditions match.


## Save

In [5]:
out_dir = PRECOMPUTED_DIR / "tables"
out_dir.mkdir(exist_ok=True, parents=True)
out_path = out_dir / "table2_gears_matched_n.csv"
table2.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
table2


Saved: precomputed/tables/table2_gears_matched_n.csv


,Condition,GEARS (n=7),CPA matched (n=7),Verdict,CPA n=100
0,K562 random,mixed (RPSA; 0.073),mixed (POLR3A; 0.138),match,single-driver (MED12; 0.163)
1,K562 stratified,mixed (SMC1A; 0.129),single-driver (MED17; 0.217),differ,single-driver (MED17; 0.163)
2,RPE1 random,mixed (BET1; 0.092),mixed (-; 0.102),match,mixed (SF3B2; 0.088)
3,RPE1 stratified,mixed (SF3B2; 0.076),mixed (SF3B2; 0.067),match,mixed (SF3B2; 0.092)
